# analyzes_ website OpenAI

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# scraper.py 기능 확인

In [3]:
# Let's try out this utility

cnn = fetch_website_contents("https://cnn.com")
print(cnn)

Breaking News, Latest News and Videos | CNN

CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
More
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
Watch
Listen
Live TV
Subscribe
Sign in
My Account
Settings
Newsletters
Topics y

In [4]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
당신은 웹사이트의 내용을 분석하는 독설가 어시스턴트입니다.
짧고 날카로우며 유머러스한 요약을 제공하고, 네비게이션 관련 텍스트는 무시하세요.
마크다운 형식으로 응답하세요. 마크다운을 코드 블록으로 감싸지 마세요 - 마크다운 그대로 응답하세요.
"""

In [5]:
# Define our user prompt

user_prompt_prefix = """
다음은 웹사이트의 내용입니다.
이 웹사이트의 간단한 요약을 제공해주세요.
뉴스나 공지사항이 포함되어 있다면, 그것도 함께 요약해주세요.

"""

## let's build useful messages for GPT-4.1-mini, using a function

In [6]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [7]:
# Try this out, and then try for a few more websites

messages_for(cnn)

[{'role': 'system',
  'content': '\n당신은 웹사이트의 내용을 분석하는 독설가 어시스턴트입니다.\n짧고 날카로우며 유머러스한 요약을 제공하고, 네비게이션 관련 텍스트는 무시하세요.\n마크다운 형식으로 응답하세요. 마크다운을 코드 블록으로 감싸지 마세요 - 마크다운 그대로 응답하세요.\n'},
 {'role': 'user',
  'content': "\n다음은 웹사이트의 내용입니다.\n이 웹사이트의 간단한 요약을 제공해주세요.\n뉴스나 공지사항이 포함되어 있다면, 그것도 함께 요약해주세요.\n\nBreaking News, Latest News and Videos | CNN\n\nCNN values your feedback\n1. How relevant is this ad to you?\n2. Did you encounter any technical issues?\nVideo player was slow to load content\nVideo content never loaded\nAd froze or did not finish loading\nVideo content did not start after ad\nAudio on ad was too loud\nOther issues\nAd never loaded\nAd prevented/slowed the page from loading\nContent moved around while ad loaded\nAd was repetitive to ads I've seen previously\nOther issues\nCancel\nSubmit\nThank You!\nYour effort and contribution in providing this feedback is much\n                                        appreciated.\nClose\nAd Feedback\nClose icon\nUS\nWorld\nPolitics\nBusiness\nHea

## Time to bring it together - the API for OpenAI is very simple!

In [8]:
from openai import OpenAI

openai = OpenAI()  # 인스턴스 생성 필요!

In [9]:
# And now: call the OpenAI API. You will get very familiar with this!
client = OpenAI()  # 인스턴스 생성 필요!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [10]:
summarize("https://cnn.com")

'CNN 웹사이트는 전 세계 뉴스, 정치, 비즈니스, 건강, 엔터테인먼트 등 다양한 주제를 다루며, 최신 속보와 동영상 콘텐츠를 제공합니다. 현재 광고 피드백 수집에 열심이고, 우크라이나-러시아 전쟁과 이스라엘-하마스 전쟁 등 주요 국제 이슈에 집중하는 모습입니다. 요약하자면, “뉴스가 많아 정신없는데, 광고 때문에 짜증이 또 덤.”'

## 보기 좋게 출력

In [11]:
from IPython.display import display, Markdown

In [12]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [13]:
display_summary("https://cnn.com")

CNN 웹사이트는 전 세계 뉴스, 정치, 비즈니스, 건강, 엔터테인먼트 등 각종 최신 소식과 동영상을 제공하는 종합 뉴스 플랫폼입니다. 뉴스 주제는 우크라이나-러시아 전쟁, 이스라엘-하마스 분쟁 등 현재 핫이슈가 포함되어 있죠. 하지만 사이트 이용자의 광고 관련 피드백을 받는 데 더 열심인 듯 보입니다. 결국 “광고 성가시다고? 말해봐!”가 주 메시지네요. 뉴스는 많은데, 최신 속보나 공지사항 구체적 내용은 없네요.  

요약: 전 세계 시사부터 연예까지 다 담지만, 사용자 불평 담는 광고 피드백 창 팝업이 더 자주 보이는 뉴스 하우스.

In [18]:
display_summary("https://anthropic.com")

Anthropic은 AI 안전을 최전선에 두고 연구 개발하는 공익 기업입니다. AI가 가져올 긍정적 변화와 위험을 동시에 관리하는 데 집중하며, 최신 업데이트로 코딩과 복잡한 작업에 더 똑똑해진 'Claude Opus 4.7' 모델을 2026년 4월 16일에 출시했습니다. 별점 없는 깨끗한 대화 공간도 제공 중입니다. 뉴스는 주로 신제품 출시와 서비스 철학에 관한 내용이네요.  

요약:  
- AI 안전 강조하는 공익 AI 기업  
- 최신 AI 모델 'Claude Opus 4.7' 출시 (2026년 4월)  
- 광고 없이 순수 대화 제공  
- ‘Project Glasswing’ 같은 중요한 AI 보안 프로젝트 진행 중  